# Peetre decomposition with psiop - updated notebook

This notebook is aligned with the current `psiop.py` API. It demonstrates and tests:

1. Symbolic Peetre decomposition: local, separable, joint.
2. Exact `apply_peetre(..., apply_joint=True)` against direct `apply(...)`.
3. Fast approximate `apply_peetre(..., apply_joint=False)`.
4. Taylor refinement and numerical convergence.
5. Asymptotic refinement, with symbolic regression checks.
6. 2D decomposition and application.
7. Timing for local variable-coefficient operators.

Important: when you compute a decomposition with non-default options, pass it to `apply_peetre` through the `decomposition` argument. Otherwise `apply_peetre` may recompute a default decomposition and the Taylor/asymptotic options will not be used.


In [ ]:
%matplotlib inline
import warnings
import time
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from psiop import PseudoDifferentialOperator


In [ ]:
def relative_error(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(b) + 1e-16)


def make_1d_periodic_grid(N=256, L=2*np.pi):
    x = np.linspace(-L/2, L/2, N, endpoint=False)
    dx = x[1] - x[0]
    kx = 2*np.pi*np.fft.fftfreq(N, d=dx)
    return x, kx


def make_2d_periodic_grid(Nx=32, Ny=32, Lx=2*np.pi, Ly=2*np.pi):
    x = np.linspace(-Lx/2, Lx/2, Nx, endpoint=False)
    y = np.linspace(-Ly/2, Ly/2, Ny, endpoint=False)

    dx = x[1] - x[0]
    dy = y[1] - y[0]

    kx = 2*np.pi*np.fft.fftfreq(Nx, d=dx)
    ky = 2*np.pi*np.fft.fftfreq(Ny, d=dy)

    X, Y = np.meshgrid(x, y, indexing='ij')

    return x, y, kx, ky, X, Y


## 1. Symbolic Peetre decomposition in 1D

We use a symbol containing:

- a local polynomial part,
- a separable/nonlocal part after Taylor refinement,
- a genuinely joint part.

The decomposition should satisfy:

```text
symbol = local_symbol + separable_symbol + joint_symbol
```


In [ ]:
x, xi = sp.symbols('x xi', real=True)

p1 = (
    (1 + x**2) * xi**2
    + x * xi
    + sp.sin(x * xi)
    + sp.exp(-(x - xi)**2)
)

op1 = PseudoDifferentialOperator(
    expr=p1,
    vars_x=[x],
    mode='symbol',
)

op1.print_peetre_decomposition(
    taylor_order=4,
    taylor_x0=[0],
    asymptotic_order=None,
    refinement_sequence=('taylor',),
)

deco1 = op1.peetre_decomposition(
    taylor_order=4,
    taylor_x0=[0],
    asymptotic_order=None,
    refinement_sequence=('taylor',),
    use_cache=False,
)

reconstructed1 = (
    deco1['local_symbol']
    + deco1['separable_symbol']
    + deco1['joint_symbol']
)

reconstruction_error1 = sp.simplify(sp.expand(reconstructed1 - p1))

print('Reconstruction error:')
print(reconstruction_error1)

assert reconstruction_error1 == 0 or bool(reconstruction_error1.equals(0))


## 2. Numerical application in 1D

We compare:

1. direct application with `op.apply(...)`;
2. exact Peetre application with `apply_joint=True`;
3. approximate Peetre application with `apply_joint=False`.

For exact mathematical comparisons, it is safer to use:

```python
freq_window=None
clamp=large_value
```

so that no numerical stabilization filter changes the operator.


In [ ]:
p_apply = (
    (1 + x**2) * xi**2
    + x * xi
    + sp.sin(x * xi)
)

op_apply = PseudoDifferentialOperator(
    expr=p_apply,
    vars_x=[x],
    mode='symbol',
)

x_grid, kx = make_1d_periodic_grid(N=512, L=2*np.pi)
u = np.exp(-x_grid**2) * np.cos(2*x_grid)

common = dict(
    boundary_condition='periodic',
    freq_window=None,
    clamp=1e12,
)

deco_apply = op_apply.peetre_decomposition(
    taylor_order=4,
    taylor_x0=[0],
    refinement_sequence=('taylor',),
    use_cache=False,
)

v_full = op_apply.apply(
    u,
    x_grid,
    kx,
    **common,
)

v_exact = op_apply.apply_peetre(
    u,
    x_grid,
    kx,
    decomposition=deco_apply,
    apply_joint=True,
    **common,
)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    v_approx = op_apply.apply_peetre(
        u,
        x_grid,
        kx,
        decomposition=deco_apply,
        apply_joint=False,
        **common,
    )

err_exact = relative_error(v_exact, v_full)
err_approx = relative_error(v_approx, v_full)

print('Exact Peetre vs direct apply:      ', err_exact)
print('Approximate Peetre vs direct apply:', err_approx)

assert err_exact < 1e-6


## 3. Purely separable nonlocal symbol

A symbol of the form

```text
a(x) q(xi)
```

should be detected as separable. The Peetre application is then especially efficient because it applies the Fourier multiplier `q(D)` once and then multiplies by `a(x)`.


In [ ]:
p_sep = sp.exp(-x**2) * sp.sqrt(xi**2 + 1)

op_sep = PseudoDifferentialOperator(
    expr=p_sep,
    vars_x=[x],
    mode='symbol',
)

op_sep.print_peetre_decomposition()

deco_sep = op_sep.peetre_decomposition(use_cache=False)

x_grid_sep, kx_sep = make_1d_periodic_grid(N=512, L=4*np.pi)
u_sep = np.exp(-x_grid_sep**2) * np.cos(5*x_grid_sep)

common_sep = dict(
    boundary_condition='periodic',
    freq_window=None,
    clamp=1e12,
)

v_full_sep = op_sep.apply(
    u_sep,
    x_grid_sep,
    kx_sep,
    **common_sep,
)

v_peetre_sep = op_sep.apply_peetre(
    u_sep,
    x_grid_sep,
    kx_sep,
    decomposition=deco_sep,
    apply_joint=True,
    **common_sep,
)

err_sep = relative_error(v_peetre_sep, v_full_sep)

print('Separable Peetre vs direct apply:', err_sep)

assert err_sep < 1e-10


## 4. Reusing a decomposition

If the operator is fixed and many input fields must be processed, compute the decomposition once and pass it through the `decomposition` argument.

This avoids repeated symbolic work and is the recommended workflow for time stepping.


In [ ]:
u1 = np.exp(-x_grid**2)
u2 = np.exp(-10*x_grid**2) * np.cos(8*x_grid)
u3 = 1 / np.cosh(x_grid)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    v1 = op_apply.apply_peetre(
        u1,
        x_grid,
        kx,
        decomposition=deco_apply,
        apply_joint=False,
        **common,
    )

    v2 = op_apply.apply_peetre(
        u2,
        x_grid,
        kx,
        decomposition=deco_apply,
        apply_joint=False,
        **common,
    )

    v3 = op_apply.apply_peetre(
        u3,
        x_grid,
        kx,
        decomposition=deco_apply,
        apply_joint=False,
        **common,
    )

print('Norm v1:', np.linalg.norm(v1))
print('Norm v2:', np.linalg.norm(v2))
print('Norm v3:', np.linalg.norm(v3))


## 5. Taylor refinement tests

This section tests Taylor refinement both symbolically and numerically.

We use a genuinely joint but smooth symbol:

```text
p(x, xi) = sqrt(1 + xi^2 + x^2)
```

A Taylor expansion in `x` around `0` produces separable terms such as:

```text
sqrt(1 + xi^2)
x^2 / (2 sqrt(1 + xi^2))
...
```

For an input field concentrated near the expansion point, the approximate Peetre application should usually improve as the Taylor order increases.


In [ ]:
p_taylor = sp.sqrt(1 + xi**2 + x**2)

op_taylor = PseudoDifferentialOperator(
    expr=p_taylor,
    vars_x=[x],
    mode='symbol',
)

x_grid_t, kx_t = make_1d_periodic_grid(N=512, L=4*np.pi)
u_taylor = np.exp(-x_grid_t**2)

common_taylor = dict(
    boundary_condition='periodic',
    freq_window=None,
    clamp=1e12,
)

v_ref_taylor = op_taylor.apply(
    u_taylor,
    x_grid_t,
    kx_t,
    **common_taylor,
)

orders = [0, 2, 4, 6]
errors_taylor = []

for order in orders:
    deco_t = op_taylor.peetre_decomposition(
        taylor_order=order,
        taylor_x0=[0],
        refinement_sequence=('taylor',),
        use_cache=False,
    )

    rec_t = (
        deco_t['local_symbol']
        + deco_t['separable_symbol']
        + deco_t['joint_symbol']
    )

    err_rec_t = sp.simplify(sp.expand(rec_t - p_taylor))
    assert err_rec_t == 0 or bool(err_rec_t.equals(0))

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        v_t = op_taylor.apply_peetre(
            u_taylor,
            x_grid_t,
            kx_t,
            decomposition=deco_t,
            apply_joint=False,
            **common_taylor,
        )

    err_t = relative_error(v_t, v_ref_taylor)
    errors_taylor.append(err_t)

    print(f'Taylor order {order}: relative error = {err_t:.6e}')

plt.figure(figsize=(7, 4))
plt.semilogy(orders, errors_taylor, 'o-')
plt.xlabel('Taylor order')
plt.ylabel('Relative error')
plt.title('Peetre Taylor approximation error')
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.show()

if errors_taylor[-1] < errors_taylor[0]:
    print('Taylor refinement improves the approximation for this test.')
else:
    print('Taylor refinement did not improve this test; check support/frequency content.')


## 6. Asymptotic refinement tests

This section tests the asymptotic refinement path.

The symbolic check is the most robust one: the decomposition should still reconstruct the original symbol exactly.

Numerical asymptotic approximations are more delicate because they are high-frequency approximations and may introduce singular terms such as `1/xi`. Therefore the numerical part below is only illustrative and is not asserted.


In [ ]:
p_asym = (
    (1 + x**2) * xi**2
    + xi / (xi**2 + x**2 + 1)
)

op_asym = PseudoDifferentialOperator(
    expr=p_asym,
    vars_x=[x],
    mode='symbol',
)

op_asym.print_peetre_decomposition(
    asymptotic_order=5,
    refine_joint=True,
    refinement_sequence=('asymptotic',),
)

deco_asym = op_asym.peetre_decomposition(
    asymptotic_order=5,
    refine_joint=True,
    refinement_sequence=('asymptotic',),
    use_cache=False,
)

rec_asym = (
    deco_asym['local_symbol']
    + deco_asym['separable_symbol']
    + deco_asym['joint_symbol']
)

err_asym = sp.simplify(sp.expand(rec_asym - p_asym))

print('Asymptotic reconstruction error:')
print(err_asym)

assert err_asym == 0 or bool(err_asym.equals(0))

print('Number of separable asymptotic terms:', len(deco_asym['separable']))
print('Number of joint residual terms:      ', len(deco_asym['joint_residual']))

# Optional numerical illustration. No strict assertion is made because
# asymptotic expansions are high-frequency approximations and may require
# windowing/clamping.
x_grid_asym, kx_asym = make_1d_periodic_grid(N=512, L=4*np.pi)
u_asym = np.exp(-x_grid_asym**2)

common_asym = dict(
    boundary_condition='periodic',
    freq_window='gaussian',
    clamp=1e3,
)

v_full_asym = op_asym.apply(
    u_asym,
    x_grid_asym,
    kx_asym,
    **common_asym,
)

v_exact_asym = op_asym.apply_peetre(
    u_asym,
    x_grid_asym,
    kx_asym,
    decomposition=deco_asym,
    apply_joint=True,
    **common_asym,
)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    v_approx_asym = op_asym.apply_peetre(
        u_asym,
        x_grid_asym,
        kx_asym,
        decomposition=deco_asym,
        apply_joint=False,
        **common_asym,
    )

print('Asymptotic exact Peetre vs direct apply:      ', relative_error(v_exact_asym, v_full_asym))
print('Asymptotic approximate Peetre vs direct apply:', relative_error(v_approx_asym, v_full_asym))


## 7. 2D Peetre decomposition and application

The same ideas extend to two spatial variables.

The grid is kept small here because the full 2D direct application can be expensive.


In [ ]:
y, eta = sp.symbols('y eta', real=True)

p2 = (
    (1 + x**2) * xi**2
    + y**2 * eta**2
    + sp.sin(x * xi + y * eta)
)

op2 = PseudoDifferentialOperator(
    expr=p2,
    vars_x=[x, y],
    mode='symbol',
)

op2.print_peetre_decomposition(
    taylor_order=2,
    taylor_x0=[0, 0],
    refinement_sequence=('taylor',),
)

deco2 = op2.peetre_decomposition(
    taylor_order=2,
    taylor_x0=[0, 0],
    refinement_sequence=('taylor',),
    use_cache=False,
)

rec2 = (
    deco2['local_symbol']
    + deco2['separable_symbol']
    + deco2['joint_symbol']
)

err2 = sp.simplify(sp.expand(rec2 - p2))

print('2D reconstruction error:')
print(err2)

assert err2 == 0 or bool(err2.equals(0))

x_grid2, y_grid2, kx2, ky2, X2, Y2 = make_2d_periodic_grid(Nx=32, Ny=32)
u2 = np.exp(-(X2**2 + Y2**2))

common2 = dict(
    boundary_condition='periodic',
    y_grid=y_grid2,
    ky=ky2,
    freq_window=None,
    clamp=1e12,
)

v2_full = op2.apply(
    u2,
    x_grid2,
    kx2,
    **common2,
)

v2_exact = op2.apply_peetre(
    u2,
    x_grid2,
    kx2,
    decomposition=deco2,
    apply_joint=True,
    **common2,
)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    v2_approx = op2.apply_peetre(
        u2,
        x_grid2,
        kx2,
        decomposition=deco2,
        apply_joint=False,
        **common2,
    )

err2_exact = relative_error(v2_exact, v2_full)
err2_approx = relative_error(v2_approx, v2_full)

print('2D exact Peetre vs direct apply:      ', err2_exact)
print('2D approximate Peetre vs direct apply:', err2_approx)

assert err2_exact < 1e-6


## 8. Optional: detailed inspection of typical benchmark symbols

These are the kinds of symbols used in solver benchmarks:

- local constant-coefficient;
- local variable-coefficient;
- nonlocal constant-coefficient;
- nonlocal variable-coefficient.


In [ ]:
benchmark_symbols = {
    '2D Local (Const)': xi**2 + eta**2,
    '2D Local (Var)': (
        1 + sp.Rational(1, 2) * sp.sin(x) * sp.cos(y)
    ) * (xi**2 + eta**2),
    '2D Non-Local (Const)': (xi**2 + eta**2)**sp.Rational(3, 4),
    '2D Non-Local (Var)': (
        1 + sp.Rational(1, 2) * sp.sin(x) * sp.cos(y)
    ) * (xi**2 + eta**2)**sp.Rational(3, 4),
}

for name, sym in benchmark_symbols.items():
    print('=' * 80)
    print('Operator:', name)
    print('=' * 80)

    op_b = PseudoDifferentialOperator(
        expr=sym,
        vars_x=[x, y],
        mode='symbol',
    )

    op_b.print_peetre_decomposition()

    deco_b = op_b.peetre_decomposition(use_cache=False)

    rec_b = (
        deco_b['local_symbol']
        + deco_b['separable_symbol']
        + deco_b['joint_symbol']
    )

    err_b = sp.simplify(sp.expand(rec_b - sym))

    print('Reconstruction error:', err_b)
    assert err_b == 0 or bool(err_b.equals(0))
    print()


## 9. Timing: direct application vs Peetre application for a local operator

For local variable-coefficient operators, Peetre application is usually much faster because each term is applied as:

```text
a(x) q(D) u = a(x) * IFFT(q(k) FFT(u))
```

instead of using the fully space-dependent Kohn-Nirenberg quadrature path.


In [ ]:
p_time = (1 + x**2) * xi**2

op_time = PseudoDifferentialOperator(
    expr=p_time,
    vars_x=[x],
    mode='symbol',
)

x_grid_time, kx_time = make_1d_periodic_grid(N=2048, L=4*np.pi)
u_time = np.exp(-x_grid_time**2)

common_time = dict(
    boundary_condition='periodic',
    freq_window=None,
    clamp=1e12,
)

deco_time = op_time.peetre_decomposition(use_cache=False)

t0 = time.perf_counter()
v_time_full = op_time.apply(
    u_time,
    x_grid_time,
    kx_time,
    **common_time,
)
t1 = time.perf_counter()

t2 = time.perf_counter()
v_time_peetre = op_time.apply_peetre(
    u_time,
    x_grid_time,
    kx_time,
    decomposition=deco_time,
    apply_joint=True,
    **common_time,
)
t3 = time.perf_counter()

err_time = relative_error(v_time_peetre, v_time_full)

print(f'Full apply time:   {t1 - t0:.6f} s')
print(f'Peetre apply time: {t3 - t2:.6f} s')
print(f'Relative error:    {err_time:.6e}')

assert err_time < 1e-10


## 10. Practical guide

Recommended usage:

1. Compute and inspect the decomposition:

```python
deco = op.peetre_decomposition(...)
op.print_peetre_decomposition(...)
```

2. For production runs, pass the precomputed decomposition:

```python
v = op.apply_peetre(u, x_grid, kx, decomposition=deco, ...)
```

3. Use `apply_joint=True` when accuracy is required.

4. Use `apply_joint=False` for fast local-plus-separable approximations.

5. Use `freq_window=None` and a large `clamp` for exact mathematical comparisons.

6. Use `freq_window='gaussian'` and a moderate `clamp` for singular asymptotic terms or for stabilized production computations.

7. If you use Weyl quantization, direct `apply` uses the Weyl-to-Kohn-Nirenberg correction. `apply_peetre` decomposes the stored symbol. For Weyl operators, verify carefully whether you need to decompose the effective Kohn-Nirenberg symbol instead of the raw Weyl symbol.
